# 07 — Inactivity Detection & Future Churn Watchlist
Goal: Identify inactive customers using computed activity scores and derive a watchlist for future churn.

In [1]:
import pandas as pd, numpy as np, sqlite3
import plotly.express as px, plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import MinMaxScaler
import warnings; warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)

In [2]:
# ── Load Datasets ──
cust = pd.read_csv('../../data/processed/customer_with_segments.csv')
bank = pd.read_csv('../../data/processed/bank_transaction_clean.csv', parse_dates=['TransactionDate','PreviousTransactionDate'])
print(f"Customer dataset: {cust.shape}")
print(f"Bank transaction dataset: {bank.shape}")

Customer dataset: (10127, 26)
Bank transaction dataset: (2512, 18)


In [3]:
# ── PART A: Customer-level inactivity ──
mms = MinMaxScaler()
inact_features = ['Months_Inactive_12_mon','Total_Trans_Ct','Total_Trans_Amt','Avg_Utilization_Ratio']
normed = pd.DataFrame(mms.fit_transform(cust[inact_features]), columns=inact_features)

cust['activity_score'] = (
    0.35 * (1 - normed['Months_Inactive_12_mon']) +
    0.30 * normed['Total_Trans_Ct'] +
    0.25 * normed['Total_Trans_Amt'] +
    0.10 * normed['Avg_Utilization_Ratio']
).round(4)

def assign_category(score):
    if score >= 0.70: return 'Active'
    elif score >= 0.45: return 'Moderately Active'
    elif score >= 0.25: return 'Inactive'
    else: return 'High Risk'

cust['activity_category'] = cust['activity_score'].apply(assign_category)

cust['future_churn_candidate'] = ((cust['Months_Inactive_12_mon'] >= 4) & (cust['Total_Trans_Ct'] < 40)).astype(int)

print("Activity category distribution:")
print(cust['activity_category'].value_counts())
print(f"\nFuture churn candidates: {cust['future_churn_candidate'].sum()} ({cust['future_churn_candidate'].mean()*100:.1f}%)")
print(f"\nActivity score stats:\n{cust['activity_score'].describe()}")

Activity category distribution:
activity_category
Inactive             5779
Moderately Active    3598
High Risk             497
Active                253
Name: count, dtype: int64

Future churn candidates: 135 (1.3%)

Activity score stats:
count   10127.0000
mean        0.4227
std         0.1179
min         0.0306
25%         0.3409
50%         0.4155
75%         0.4933
max         0.8285
Name: activity_score, dtype: float64


In [4]:
# ── PART B: Bank transaction inactivity ──
bank['TransactionDate'] = pd.to_datetime(bank['TransactionDate'])
bank['PreviousTransactionDate'] = pd.to_datetime(bank['PreviousTransactionDate'])
bank['days_since_last_txn'] = (bank['TransactionDate'] - bank['PreviousTransactionDate']).dt.days.abs()

account_summary = bank.groupby('AccountID').agg(
    latest_txn=('TransactionDate','max'),
    total_txns=('TransactionID','count'),
    total_amount=('TransactionAmount','sum'),
    avg_amount=('TransactionAmount','mean'),
    avg_days_gap=('days_since_last_txn','mean'),
    max_days_gap=('days_since_last_txn','max'),
    login_attempts_avg=('LoginAttempts','mean'),
    balance=('AccountBalance','last')
).reset_index()

ref_date = bank['TransactionDate'].max()
account_summary['days_since_last_txn'] = (ref_date - account_summary['latest_txn']).dt.days
account_summary['bank_inactive_flag'] = (account_summary['days_since_last_txn'] > 30).astype(int)

print(f"\nBank accounts with >30 day gap: {account_summary['bank_inactive_flag'].sum()}")
print(f"Days since last txn distribution:")
print(account_summary['days_since_last_txn'].describe().round(2))


Bank accounts with >30 day gap: 322
Days since last txn distribution:
count   495.0000
mean     68.4800
std      68.6400
min       0.0000
25%      20.0000
50%      47.0000
75%      95.5000
max     363.0000
Name: days_since_last_txn, dtype: float64


In [5]:
# ── SQL Analysis ──
con = sqlite3.connect(':memory:')
cust.to_sql('cust', con, index=False, if_exists='replace')
bank.to_sql('bank', con, index=False, if_exists='replace')

q1 = pd.read_sql_query("""
    SELECT activity_category,
           COUNT(*) as count,
           SUM(is_attrited) as churned,
           ROUND(AVG(is_attrited)*100,2) as churn_pct,
           ROUND(AVG(activity_score),4) as avg_score
    FROM cust GROUP BY activity_category ORDER BY churn_pct DESC
""", con)
print("Q1: Activity Category vs Churn\n", q1.to_string(index=False), "\n")

q2 = pd.read_sql_query("""
    SELECT segment, COUNT(*) as total_in_segment,
           SUM(future_churn_candidate) as watchlist_count,
           ROUND(100.0*SUM(future_churn_candidate)/COUNT(*),2) as watchlist_pct
    FROM cust GROUP BY segment ORDER BY watchlist_pct DESC
""", con)
print("Q2: Watchlist by Segment\n", q2.to_string(index=False), "\n")

q3 = pd.read_sql_query("""
    SELECT future_churn_candidate,
           COUNT(*) as count,
           ROUND(AVG(is_attrited)*100,2) as actual_churn_rate
    FROM cust GROUP BY future_churn_candidate
""", con)
print("Q3: Churn Validation of Flag\n", q3.to_string(index=False), "\n")

cust['score_decile'] = pd.qcut(cust['activity_score'], 10, labels=[f'D{i}' for i in range(1,11)])
decile_churn = cust.groupby('score_decile', observed=True).agg(
    count=('CLIENTNUM','count'),
    churn_rate=('is_attrited','mean')
).reset_index()
decile_churn['churn_rate'] = (decile_churn['churn_rate']*100).round(2)
print("Q4: Activity Score Decile vs Churn Rate\n", decile_churn.to_string(index=False), "\n")

q5 = pd.read_sql_query("""
    SELECT Channel,
           ROUND(AVG(days_since_last_txn),1) as avg_gap,
           COUNT(*) as count
    FROM bank GROUP BY Channel
""", con)
print("Q5: Bank Channel Inactivity\n", q5.to_string(index=False))

Q1: Activity Category vs Churn
 activity_category  count  churned  churn_pct  avg_score
        High Risk    497      233    46.8800     0.2057
         Inactive   5779     1266    21.9100     0.3626
Moderately Active   3598      128     3.5600     0.5267
           Active    253        0     0.0000     0.7423 

Q2: Watchlist by Segment
           segment  total_in_segment  watchlist_count  watchlist_pct
At-Risk Customers              2451              106         4.3200
Premium Customers              1194               18         1.5100
     Deal Hunters              2883               11         0.3800
     Silent Users              2730                0         0.0000
   Daily Spenders               869                0         0.0000 

Q3: Churn Validation of Flag
  future_churn_candidate  count  actual_churn_rate
                      0   9992            15.5300
                      1    135            55.5600 

Q4: Activity Score Decile vs Churn Rate
 score_decile  count  churn_

In [6]:
# ── Visualizations ──
# Plot 1
color_map = {'Active':'#006FCF', 'Moderately Active':'#C07000', 'Inactive':'#E88B00', 'High Risk':'#C0001A'}
fig1 = px.bar(q1, x='activity_category', y='count', color='activity_category', color_discrete_map=color_map, title="Activity Category Distribution")
fig1.show()

# Plot 2
fig2 = px.bar(q1, x='activity_category', y='churn_pct', color='activity_category', color_discrete_map=color_map, title="Activity Category Validation Against Real Churn (%)")
fig2.show()

# Plot 3
fig3 = px.histogram(cust, x='activity_score', color='activity_category', color_discrete_map=color_map, title="Activity Score Distribution")
fig3.add_vline(x=0.25, line_dash="dash", annotation_text="0.25")
fig3.add_vline(x=0.45, line_dash="dash", annotation_text="0.45")
fig3.add_vline(x=0.70, line_dash="dash", annotation_text="0.70")
fig3.show()

# Plot 4
fig4 = px.bar(q2, x='segment', y='watchlist_pct', title="Future Churn Watchlist % by Customer Segment")
fig4.show()

# Plot 5
fig5 = px.line(decile_churn, x='score_decile', y='churn_rate', markers=True, title="Churn Rate by Activity Score Decile (Lower score = Higher churn)")
fig5.show()

# Plot 6
fig6 = px.scatter(cust, x='activity_score', y='Total_Trans_Amt', color=cust['is_attrited'].astype(str), title="Activity Score vs Transaction Amount", opacity=0.5)
fig6.show()

# Plot 7
fig7 = px.histogram(bank, x='days_since_last_txn', title="Bank Account Transaction Gap Distribution")
fig7.add_vline(x=30, line_dash="dash", line_color="red", annotation_text="30 Days")
fig7.show()

# Plot 8
fig8 = px.scatter(cust, x='Months_Inactive_12_mon', y='activity_score', color='activity_category', color_discrete_map=color_map, title="Inactivity Months vs Computed Activity Score")
fig8.show()

## Key Insights
- Activity scoring successfully stratifies churn risk. 'High Risk' category shows massively elevated churn vs baseline.
- The `future_churn_candidate` flag is highly correlated with actual historical churn.
- Bank dataset transaction gaps highlight accounts that might need re-engagement campaigns.
- Watchlist segment distributions inform which marketing strategies to employ.

In [7]:
# ── Save ──
cust.to_csv('../../data/processed/inactivity_scores.csv', index=False)
account_summary.to_csv('../../data/processed/bank_account_inactivity.csv', index=False)
print(f"Saved inactivity_scores.csv — shape {cust.shape}")
print(f"Future churn watchlist size: {cust['future_churn_candidate'].sum()}")
print(f"Watchlist as % of portfolio: {cust['future_churn_candidate'].mean()*100:.1f}%")
con.close()

Saved inactivity_scores.csv — shape (10127, 30)
Future churn watchlist size: 135
Watchlist as % of portfolio: 1.3%
